In [1]:
import os
import gc

import numpy as np
import pandas as pd

import torch
from transformers import pipeline
from transformers import AutoTokenizer, AutoConfig
from transformers import DataCollatorForLanguageModeling
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

import evaluate
import datasets
from datasets import ClassLabel, load_dataset, Dataset, DatasetDict, load_metric

import warnings
warnings.filterwarnings('ignore')

In [2]:
class CFG:
    wandb = False
    report_to = None
    lab_assignment = 5
    _wandb_kernel = "temuujin"

    debug = False
    num_workers = 12

    tokenizer_name = 'bayartsogt/mongolian-gpt2'
    model_name = 'bayartsogt/mongolian-gpt2'

    project = 'NUM-Machine-Learning-Lab-5'
    name = "Lab 5"

    config = {
        "output_dir": "lab5_finetune_gpt2",
        "group": model_name,
        "learning_rate": 2e-5,
        "weight_decay": 1e-2,
        'num_train_epochs': 3,
        "train_batch_size": 32,
        "eval_batch_size": 32,
        "dataloader_num_workers": num_workers,
        "finetuning_task": 'ner',
        "evaluation_strategy": 'epoch',
        "logging_strategy": 'epoch',
        "overwrite_output_dir": True,
        "push_to_hub": False,
    }

    model_save_dir = "lab5_gpt2"

    test_size = 0.2

    train = True
    eval = True

    eval_metric = "seqeval"

    early_stopping_patience = 15

if CFG.debug:
    CFG.config['num_train_epochs'] = 2

if CFG.wandb:
    os.environ["WANDB_SILENT"] = "True"
    CFG.report_to = "wandb"

    import wandb
    wandb.login()

    run = wandb.init(
        project = CFG.project,
        name = CFG.name,
        config = CFG.config
    )

config = CFG.config

In [3]:
def concatenate_columns(example):
    example["prompt"] = example["prompt"] + " " + example["answer"]
    return example

In [4]:
df = pd.read_csv('khk.noun.tsv', sep = '\t')

df['prompt'] = '<s> bb: ' + df['prompt']
df['answer'] = df['answer'] + '</s>'

infl_dataset = Dataset.from_pandas(df)
ds_train_devtest = infl_dataset.train_test_split(test_size = 0.025, seed = 42)
ds_devtest = ds_train_devtest['test'].train_test_split(test_size = 0.5, seed = 42)

ds_splits = DatasetDict({
    'train': ds_train_devtest['train'],
    'valid': ds_devtest['train'],
    'test': ds_devtest['test']
})

ds_splits["train"] = ds_splits["train"].map(concatenate_columns)
ds_splits["valid"] = ds_splits["valid"].map(concatenate_columns)

ds_splits = ds_splits.flatten()

Map:   0%|          | 0/14036 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

In [5]:
ds_splits

DatasetDict({
    train: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 14036
    })
    valid: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 180
    })
    test: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 180
    })
})

In [6]:
block_size = 64
tokenizer = AutoTokenizer.from_pretrained(CFG.tokenizer_name)

def preprocess_function(examples):
    return tokenizer(examples["prompt"])

tokenized_ds = ds_splits.map(
    preprocess_function,
    batched = True,
    num_proc = 4,
    remove_columns = ds_splits["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/14036 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

In [7]:
def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized_ds.map(group_texts, batched = True, num_proc = 4)

Map (num_proc=4):   0%|          | 0/14036 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

# Train as is

In [8]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm = False)
model = AutoModelForCausalLM.from_pretrained(CFG.model_name)

config = CFG.config

test_ver = 3
OUTPUT_MODEL = os.path.join(CFG.model_save_dir, f"test_v{test_ver}")

training_args = TrainingArguments(
    report_to = CFG.report_to,
    output_dir = OUTPUT_MODEL,
    #num_train_epochs = config["num_train_epochs"],
    #per_device_train_batch_size = config["train_batch_size"],
    #per_device_eval_batch_size = config["eval_batch_size"],
    overwrite_output_dir = config["overwrite_output_dir"],
    learning_rate = config["learning_rate"],
    weight_decay = config["weight_decay"],
    evaluation_strategy = config["evaluation_strategy"],
    push_to_hub = config["push_to_hub"],
    #do_eval = True,
    #disable_tqdm = True
)

In [9]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = lm_dataset["train"],
    eval_dataset = lm_dataset["valid"],
    tokenizer = tokenizer,
    data_collator = data_collator,
    compute_metrics = None,
)

trainer.train()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: temuujin-razy. Use `wandb login --relogin` to force relogin


  0%|          | 0/969 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

{'eval_loss': 1.5579043626785278, 'eval_runtime': 0.0871, 'eval_samples_per_second': 367.335, 'eval_steps_per_second': 45.917, 'epoch': 1.0}
{'loss': 2.6137, 'grad_norm': 1.3151301145553589, 'learning_rate': 9.680082559339526e-06, 'epoch': 1.55}


  0%|          | 0/4 [00:00<?, ?it/s]

{'eval_loss': 1.3676233291625977, 'eval_runtime': 0.0739, 'eval_samples_per_second': 433.069, 'eval_steps_per_second': 54.134, 'epoch': 2.0}


  0%|          | 0/4 [00:00<?, ?it/s]

{'eval_loss': 1.3429744243621826, 'eval_runtime': 0.076, 'eval_samples_per_second': 420.871, 'eval_steps_per_second': 52.609, 'epoch': 3.0}
{'train_runtime': 103.5732, 'train_samples_per_second': 74.788, 'train_steps_per_second': 9.356, 'train_loss': 1.9840989245718847, 'epoch': 3.0}


TrainOutput(global_step=969, training_loss=1.9840989245718847, metrics={'train_runtime': 103.5732, 'train_samples_per_second': 74.788, 'train_steps_per_second': 9.356, 'train_loss': 1.9840989245718847, 'epoch': 3.0})

In [10]:
eval_results = trainer.evaluate()
print(f"Perplexity: {np.exp(eval_results['eval_loss']):.2f}")

  0%|          | 0/4 [00:00<?, ?it/s]

Perplexity: 3.83


In [11]:
generator = pipeline("text-generation", model = model, tokenizer = tokenizer, num_beams = 5)

In [12]:
prompt = "<s> bb: сургуулийн"
ans = generator(prompt, max_length = 20)

print(str(ans[0]['generated_text']))
print(str(ans))

torch.cuda.empty_cache()
gc.collect()

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


<s> bb: сургуулийн сургууль <gen> bb: төлгийг төлгө <
[{'generated_text': '<s> bb: сургуулийн сургууль <gen> bb: төлгийг төлгө <'}]


128

# Layer freeze experiment

In [ ]:
config = AutoConfig.from_pretrained(CFG.model_name)
model = AutoModelForCausalLM.from_config(config)

print(config)
print(model)

In [ ]:
for param in model.base_model.parameters():
    param.requires_grad = True

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = lm_dataset["train"],
    eval_dataset = lm_dataset["test"],
    tokenizer = tokenizer,
    data_collator = data_collator,
)

trainer.train()

In [ ]:
eval_results = trainer.evaluate()

prompt = '<s> bb: гарагийн'
generator = pipeline("text-generation", model = model, tokenizer = tokenizer, num_beams = 5)

ans = generator(prompt, max_length = 20)

print(str(ans[0]['generated_text']))
print(str(ans))

torch.cuda.empty_cache()
gc.collect()